# Script 4: Dynamic Few-Shot Example Selection
Static few-shot prompts (hardcoding $3-5$ examples directly into the system prompt) waste valuable tokens and can degrade LLM performance when examples aren't relevant to the user's specific query.

In system design interviews, you will be asked:
  -  "How do you design an In-Context Learning system that dynamically injects the most semantically relevant few-shot examples into the context window without blowing token limits or requiring full model fine-tuning?"

# Core Concept: Vector Similarity Few-Shot Selection
Instead of static prompting:
    1. Maintain a repository of high-quality (Input, Output) example pairs.Embed the user's incoming query.
    2. Perform a semantic similarity search (Cosine / Euclidean) over the repository.
    3. Retrieve the top $K$ most relevant examples and dynamically construct the prompt before calling Groq.

In [13]:
import os
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate
from langchain_groq import ChatGroq

In [14]:
# 1. Example Bank (Store of curated golden examples)
EXAMPLES = [
    {
        "input": "Log event: User login failed 3 times from IP 192.168.1.10",
        "output": "SEVERITY: HIGH | CATEGORY: Authentication | ACTION: Block IP temporarily"
    },
    {
        "input": "Log event: Database response time spike to 4500ms on query SELECT * FROM orders",
        "output": "SEVERITY: MEDIUM | CATEGORY: Database | ACTION: Optimize query or inspect connection pool"
    },
    {
        "input": "Log event: Scheduled cron job completed successfully in 12s",
        "output": "SEVERITY: LOW | CATEGORY: System | ACTION: No action required"
    },
    {
        "input": "Log event: Memory usage reached 92% on server app-node-01",
        "output": "SEVERITY: HIGH | CATEGORY: Infrastructure | ACTION: Scale instance or restart container"
    }
]

In [15]:
# 2. Example Prompt Template
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

In [16]:
# 3. Define Few-Shot Chat Template (Static baseline or top-K selector)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=EXAMPLES[:2]  # Simulating top-2 dynamic semantic retrieval
)

In [17]:
# 4. Assemble Final System Prompt
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an automated DevOps Log Analyzer. Classify incoming logs matching the formatting style of the examples below."),
    few_shot_prompt,
    ("human", "{user_input}")
])

In [ ]:
# 5. Initialize Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key="Your_API_KEY_HERE"
)

In [19]:
chain = final_prompt | llm

In [20]:
# --- Run Live Test ---
if __name__ == "__main__":
    user_log = "Log event: Disk usage exceeded 95% threshold on /data partition"
    
    print(f"Analyzing log with Groq (Llama 3.3 70B)...")
    response = chain.invoke({"user_input": user_log})
    
    print("\n================ FEW-SHOT CLASSIFICATION RESULT ================")
    print(response.content)

Analyzing log with Groq (Llama 3.3 70B)...

================ FEW-SHOT CLASSIFICATION RESULT ================
SEVERITY: HIGH | CATEGORY: Storage | ACTION: Alert admin and free up space on /data partition


### Architectural Trade-Off Matrix (Interview Quick Reference)

| Strategy | Latency | Cost | Maintenance / Upkeep | Best Used For |
| :--- | :--- | :--- | :--- | :--- |
| **Zero-Shot Prompting** | Lowest | Lowest | Zero setup | Standard instructions / simple tasks |
| **Dynamic Few-Shot (RAG)** | Medium | Low-Medium | High quality examples in vector DB | Unstructured outputs, specialized formats |
| **Fine-Tuning (LoRA/PEFT)** | Lowest (Inference) | High (Training) | Retraining pipelines required | Style alignment, small parameter models |